<a href="https://colab.research.google.com/github/Salazar254/meme_coin/blob/main/retrain_rug_model_2025_2026_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install onnxruntime torch numpy pandas scikit-learn requests tqdm pyarrow xgboost skl2onnx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 32.6 MB/s eta 0:00:00


In [16]:
import subprocess
import shutil
import os

# Clone the repository
repo_path = '/content/repo'
if not os.path.exists(repo_path):
    print("Cloning meme_coin repository...")
    subprocess.run(
        ['git', 'clone', 'https://github.com/Salazar254/meme_coin', repo_path],
        check=True
    )
    print("Repository cloned.")
else:
    print("Repository already cloned.")

# Define the CSV files to copy
csv_files = [
    'confirmed_rugs.csv',
    'known_survivors.csv',
    'known_rugger_deployers.csv',
    'rug_mechanics_features.csv',
    'pump_fun_survival_stats.csv',
]

# Ensure the target directory exists in Google Drive
drive_base = CONFIG['DRIVE_BASE']
Path(drive_base).mkdir(parents=True, exist_ok=True)

print(f"Copying CSV files to Drive directory: {drive_base}")
for f in csv_files:
    src = f'{repo_path}/ml/data/{f}'
    dst = f'{drive_base}/{f}'
    if os.path.exists(src):
        if not os.path.exists(dst) or (os.path.getsize(src) != os.path.getsize(dst)): # Check if file exists or is different
            shutil.copy(src, dst)
            print(f'  Copied {f} to Drive')
        else:
            print(f'  Skipped {f}, already exists and is identical in Drive')
    else:
        print(f'  WARNING: {f} not found in repository at {src}')


Cloning meme_coin repository...
Repository cloned.
Copying CSV files to Drive directory: /content/drive/MyDrive/meme_bot_training


In [41]:
from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score
import xgboost as xgb
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

CONFIG = {
    "HELIUS_API_KEY": os.environ.get("HELIUS_API_KEY", ""),
    "HELIUS_BASE_URL": "https://api.helius.xyz",
    "GRADUATION_MANIFEST_PATH": "/content/drive/MyDrive/meme_bot_training/graduated_tokens.parquet",
    "RAW_DATA_PATH": "/content/drive/MyDrive/meme_bot_training/raw_data.parquet",
    "FEATURES_PATH": "/content/drive/MyDrive/meme_bot_training/features.parquet",
    "COLLECTION_CHECKPOINT_PATH": "/content/drive/MyDrive/meme_bot_training/collection_checkpoint.json",
    "TRAIN_CACHE_PATH": "/content/drive/MyDrive/meme_bot_training/train_dataset.pt",
    "VAL_CACHE_PATH": "/content/drive/MyDrive/meme_bot_training/val_dataset.pt",
    "TEST_CACHE_PATH": "/content/drive/MyDrive/meme_bot_training/test_dataset.pt",
    "BEST_CHECKPOINT_PATH": "/content/drive/MyDrive/meme_bot_training/rug_model_best.pt",
    "ONNX_OUTPUT_PATH": "/content/drive/MyDrive/meme_bot_training/rug_model.onnx",
    "DEPLOYER_LOOKUP_PATH": "/content/drive/MyDrive/meme_bot_training/deployer_lookup.json",
    "MODEL_META_PATH": "/content/drive/MyDrive/meme_bot_training/rug_model_meta.json",
    "SEQUENCE_LENGTH": 16,
    "SEQUENCE_FEATURES": ["holders", "liquidity", "volume", "ratio", "velocity", "tx_count"],
    "TABULAR_FEATURES": [
        "rugPullRisk",
        "honeypotRisk",
        "lpBurnGap",
        "transferTaxPct",
        "topHolderPct",
        "devHoldPct",
        "mutableMetadata",
        "mintAuthority",
        "freezeAuthority",
        "volatility1m",
        "lowLiquidity",
        "lowBuyers",
        "rugcheckLpUnlocked",
        "rugcheckDangerSignals",
    ],
    "TRAIN_START": "2026-06-23T00:00:00Z",
    "TRAIN_END": "2026-06-23T06:00:00Z",
    "VAL_START": "2026-06-23T06:00:01Z",
    "VAL_END": "2026-06-23T08:00:00Z",
    "TEST_START": "2026-06-23T08:00:01Z",
    "TEST_END": pd.Timestamp.utcnow().isoformat(),
    "LABEL_WINDOW_HOURS": 72,
    "PRICE_DROP_RUG_THRESHOLD": 0.40,
    "LIQUIDITY_DROP_RUG_THRESHOLD": 0.40,
    "LOCK_BURN_THRESHOLD": 0.90,
    "LOCK_LOCKED_THRESHOLD": 90.0,
    "ENTRY_DELAY_SECONDS": 120,
    "TIME_BUCKET_MINUTES": 5,
    "MAX_SWAP_PAGES": 200,
    "SWAP_PAGE_LIMIT": 100,
    "CHECKPOINT_EVERY_TOKENS": 100,
    "HTTP_TIMEOUT_SECONDS": 30,
    "MAX_RETRIES": 6,
    "INITIAL_BACKOFF_SECONDS": 1.5,
    "BATCH_SIZE": 256,
    "LEARNING_RATE": 1e-3,
    "WEIGHT_DECAY": 0.01,
    "EPOCHS": 40,
    "PATIENCE": 5,
    "SEED": 20260622,
    "DROPOUT_1": 0.3,
    "DROPOUT_2": 0.2,
    "AUC_QUALITY_GATE": 0.65,
    "MAX_SLIPPAGE_PCT": 1.0,
    "UNKNOWN_DEPLOYER_ID": 0,
    "EXPORT_OPSET": 15,

    "XGB_N_ESTIMATORS": 300,
    "XGB_MAX_DEPTH": 5,
    "XGB_LEARNING_RATE": 0.05,
    "XGB_SUBSAMPLE": 0.8,
    "XGB_COLSAMPLE": 0.8,
    "XGB_ENSEMBLE_WEIGHT": 0.4,
    "TORCH_ENSEMBLE_WEIGHT": 0.6,
    "XGB_MODEL_OUT": "/content/drive/MyDrive/meme_bot_training/xgb_model.onnx",

    "GITHUB_REPO": "https://github.com/Salazar254/meme_coin",
    "GITHUB_TOKEN": "",
    "AUTO_PUSH": True,

    "DRIVE_BASE": "/content/drive/MyDrive/meme_bot_training",
    "CHECKPOINT_PATH": "/content/drive/MyDrive/meme_bot_training/collection_checkpoint.json",
    "RUG_MODEL_OUT": "/content/drive/MyDrive/meme_bot_training/rug_model.onnx",
    "DEPLOYER_OUT": "/content/drive/MyDrive/meme_bot_training/deployer_lookup.json",
    "META_OUT": "/content/drive/MyDrive/meme_bot_training/rug_model_meta.json",
    "FEATURE_IMP_OUT": "/content/drive/MyDrive/meme_bot_training/feature_importance.json",
    "TARGET_TOKENS": 5000
}

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Read API keys from Colab secrets
try:
    from google.colab import userdata
    helius_key = userdata.get('HELIUS_API_KEY')
    CONFIG['HELIUS_API_KEY'] = helius_key if helius_key is not None else ""
except:
    print("No HELIUS_API_KEY secret found - Helius API calls will fail")

try:
    from google.colab import userdata
    CONFIG['GITHUB_TOKEN'] = userdata.get('GITHUB_API_KEY')
except:
    print("No GITHUB_TOKEN secret found - auto-push disabled")


random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
torch.manual_seed(CONFIG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"device": str(DEVICE), "sequence_length": CONFIG["SEQUENCE_LENGTH"], "feature_count": len(CONFIG["TABULAR_FEATURES"])})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No GITHUB_TOKEN secret found - auto-push disabled
{'device': 'cuda', 'sequence_length': 16, 'feature_count': 14}


In [48]:
RAW_DATA_PATH = Path(CONFIG["RAW_DATA_PATH"])
COLLECTION_CHECKPOINT_PATH = Path(CONFIG["COLLECTION_CHECKPOINT_PATH"])


def clamp(value: float, low: float = 0.0, high: float = 1.0) -> float:
    return max(low, min(high, value))


def safe_float(value: Any, fallback: float = 0.0) -> float:
    try:
        if value is None or value == "":
            return fallback
        value = float(value)
        return value if math.isfinite(value) else fallback
    except Exception:
        return fallback


def to_timestamp(value: Any) -> pd.Timestamp:
    if isinstance(value, pd.Timestamp):
        return value.tz_convert("UTC") if value.tzinfo else value.tz_localize("UTC")
    if isinstance(value, (int, float)):
        unit = "ms" if value > 10_000_000_000 else "s"
        return pd.to_datetime(value, unit=unit, utc=True)
    return pd.to_datetime(value, utc=True)


def with_backoff(fn, *args, **kwargs):
    delay = CONFIG["INITIAL_BACKOFF_SECONDS"]
    last_error = None
    for attempt in range(CONFIG["MAX_RETRIES"]):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            last_error = exc
            if attempt == CONFIG["MAX_RETRIES"] - 1:
                break
            sleep_for = delay * (2 ** attempt)
            print(f"retry {attempt + 1}/{CONFIG['MAX_RETRIES']} after error: {exc}")
            time.sleep(sleep_for)
    raise last_error


def helius_get(path: str, params: dict[str, Any]) -> Any:
    url = f"{CONFIG['HELIUS_BASE_URL'].rstrip('/')}{path}"
    # Debug print: Check the API key value just before it's used
    print(f"DEBUG: HELIUS_API_KEY in helius_get: '{CONFIG['HELIUS_API_KEY']}'")
    params = {**params, "api-key": CONFIG["HELIUS_API_KEY"]}
    response = requests.get(url, params=params, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"])
    if response.status_code in {429, 500, 502, 503, 504}:
        raise RuntimeError(f"helius_transient_http_{response.status_code}")
    response.raise_for_status()
    return response.json()

def get_mint_list_from_dexscreener(target=500):
    """
    Fetches recently active Solana tokens from DexScreener + Helius.
    Uses NO text-search 'q' param (that filters by token NAME, not chain).
    Returns a DataFrame with columns:
    mint, graduation_timestamp, initial_liquidity_sol, deployer
    """
    HELIUS_KEY = CONFIG["HELIUS_API_KEY"]
    mints_set: set[str] = set()
    source_counts: dict[str, int] = {}
    mints_list: list[dict] = []
    fetch_target = target * 3  # collect 3x to account for filtering

    print(f"Building mint list (target >= {target} tokens)...")

    # === SOURCE 1: DexScreener pairs (NO 'q' param) ===
    print("\n[1/4] DexScreener pairs (solana/raydium, no q param)...")
    try:
        url = "https://api.dexscreener.com/latest/dex/pairs/solana/raydium"
        resp = requests.get(url, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"])
        if resp.status_code == 200:
            data = resp.json()
            pairs = data.get("pairs", [])
            cutoff_ms = int(datetime(2025, 1, 1, tzinfo=timezone.utc).timestamp() * 1000)
            count = 0
            for pair in pairs:
                base = pair.get("baseToken", {})
                addr = base.get("address", "")
                created = pair.get("pairCreatedAt", 0)
                liquidity = float(pair.get("liquidity", {}).get("usd", 0))
                if addr and created >= cutoff_ms and liquidity > 0 and not addr.startswith("0x"):
                    if addr not in mints_set:
                        mints_set.add(addr)
                        mints_list.append({
                            "mint": addr,
                            "graduation_timestamp": created / 1000,
                            "initial_liquidity_sol": liquidity / 150,
                            "deployer": "",
                        })
                        count += 1
            source_counts["dexscreener_pairs"] = count
            print(f"  DexScreener pairs: {count} tokens (total unique: {len(mints_set)})")
        else:
            print(f"  DexScreener pairs failed: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  DexScreener pairs error: {e}")

    # === SOURCE 2: DexScreener token profiles (NO 'q' param) ===
    print("\n[2/4] DexScreener token profiles...")
    try:
        url = "https://api.dexscreener.com/token-profiles/latest/v1"
        resp = requests.get(url, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"])
        count = 0
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, list):
                for token in data:
                    if token.get("chainId") == "solana":
                        addr = token.get("tokenAddress", "")
                        if addr and not addr.startswith("0x") and addr not in mints_set:
                            mints_set.add(addr)
                            mints_list.append({
                                "mint": addr,
                                "graduation_timestamp": time.time() - 86400,
                                "initial_liquidity_sol": 0,
                                "deployer": "",
                            })
                            count += 1
            source_counts["dexscreener_profiles"] = count
            print(f"  DexScreener profiles: {count} solana tokens (total unique: {len(mints_set)})")
        else:
            print(f"  DexScreener profiles failed: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  DexScreener profiles error: {e}")

    # === SOURCE 3: Helius pump.fun migration program ===
    print("\n[3/4] Helius pump.fun migration transactions...")
    PUMP_MIGRATION = "39azUYFWPz3VHgKCf3VChUwbpURdCHRxjWVowf5jUJjg"
    try:
        url = f"https://api.helius.xyz/v0/addresses/{PUMP_MIGRATION}/transactions"
        params = {"api-key": HELIUS_KEY, "limit": 100, "type": "TRANSFER"}
        resp = requests.get(url, params=params, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"])
        count = 0
        if resp.status_code == 200:
            txs = resp.json()
            for tx in txs:
                for acct in tx.get("accountData", []):
                    mint = acct.get("mint") or acct.get("account")
                    if mint and len(str(mint)) >= 32 and str(mint) not in mints_set:
                        mints_set.add(str(mint))
                        mints_list.append({
                            "mint": str(mint),
                            "graduation_timestamp": tx.get("timestamp", time.time()),
                            "initial_liquidity_sol": 0,
                            "deployer": "",
                        })
                        count += 1
            source_counts["helius_migration"] = count
            print(f"  Helius migration: {count} mints (total unique: {len(mints_set)})")
        elif resp.status_code == 429:
            print("  Helius migration rate limited (429)")
            source_counts["helius_migration"] = 0
        else:
            print(f"  Helius migration: HTTP {resp.status_code}")
            source_counts["helius_migration"] = 0
    except Exception as e:
        print(f"  Helius migration error: {e}")
        source_counts["helius_migration"] = 0

    # === SOURCE 4: Helius getAssetsByGroup (pump.fun, paginated) ===
    print("\n[4/4] Helius getAssetsByGroup (pump.fun)...")
    try:
        count = 0
        page = 1
        while len(mints_set) < fetch_target and page <= 5:
            body = {
                "jsonrpc": "2.0",
                "id": 1,
                "method": "getAssetsByGroup",
                "params": {
                    "groupKey": "collection",
                    "groupValue": "pump.fun",
                    "page": page,
                    "limit": 1000,
                },
            }
            resp = requests.post(
                f"https://mainnet.helius-rpc.com/?api-key={HELIUS_KEY}",
                json=body,
                timeout=CONFIG["HTTP_TIMEOUT_SECONDS"],
            )
            if resp.status_code == 200:
                result = resp.json().get("result", {})
                items = result.get("items", [])
                if not items:
                    break
                for item in items:
                    addr = item.get("id", "")
                    if addr and addr not in mints_set:
                        mints_set.add(addr)
                        mints_list.append({
                            "mint": addr,
                            "graduation_timestamp": time.time() - 86400,
                            "initial_liquidity_sol": 0,
                            "deployer": "",
                        })
                        count += 1
                page += 1
            elif resp.status_code == 429:
                print(f"  Helius DAS rate limited on page {page}, sleeping 10s...")
                time.sleep(10)
            else:
                print(f"  Helius DAS error: HTTP {resp.status_code}")
                break
        source_counts["helius_das"] = count
        print(f"  Helius DAS: {count} mints from {page - 1} pages (total unique: {len(mints_set)})")
    except Exception as e:
        print(f"  Helius DAS error: {e}")
        source_counts["helius_das"] = 0

    print(f"\n{'=' * 50}")
    print(f"Total unique mints collected: {len(mints_list)}")
    for src, cnt in source_counts.items():
        print(f"  {src}: {cnt}")

    if len(mints_list) < 100:
        print("WARNING: Very few mints collected. Check API keys and network connectivity.")

    # Build DataFrame, save as parquet
    df = pd.DataFrame(mints_list[:fetch_target])
    save_path = Path(CONFIG["GRADUATION_MANIFEST_PATH"])
    save_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(save_path, index=False)
    print(f"Saved manifest: {len(df)} tokens -> {save_path}")
    return df


def load_collection_checkpoint() -> dict[str, Any]:
    if COLLECTION_CHECKPOINT_PATH.exists():
        return json.loads(COLLECTION_CHECKPOINT_PATH.read_text())
    return {"processed_mints": [], "raw_rows": 0, "updated_at": None}


def save_collection_checkpoint(state: dict[str, Any]) -> None:
    state["updated_at"] = datetime.now(timezone.utc).isoformat()
    COLLECTION_CHECKPOINT_PATH.write_text(json.dumps(state, indent=2))


def fetch_swap_transactions(mint: str) -> list[dict[str, Any]]:
    before = None
    rows: list[dict[str, Any]] = []
    for _ in range(CONFIG["MAX_SWAP_PAGES"]):
        params = {"type": "SWAP", "limit": CONFIG["SWAP_PAGE_LIMIT"]}
        if before:
            params["before"] = before
        payload = with_backoff(helius_get, f"/v0/addresses/{mint}/transactions", params)
        if not payload:
            break
        rows.extend(payload)
        if len(payload) < CONFIG["SWAP_PAGE_LIMIT"]:
            break
        before = payload[-1].get("signature")
        if not before:
            break
    return rows


def normalize_swap_rows(mint_row: pd.Series, swaps: list[dict[str, Any]]) -> list[dict[str, Any]]:
    normalized: list[dict[str, Any]] = []
    for tx in swaps:
        ts = to_timestamp(tx.get("timestamp") or tx.get("blockTime") or tx.get("slotTime"))
        token_transfers = tx.get("tokenTransfers") or []
        native_transfers = tx.get("nativeTransfers") or []
        token_amount = 0.0
        sol_amount = 0.0
        buyer = None
        seller = None
        for transfer in token_transfers:
            if transfer.get("mint") == mint_row["mint"]:
                token_amount += abs(safe_float(transfer.get("tokenAmount") or transfer.get("rawTokenAmount", {}).get("tokenAmount")))
                buyer = buyer or transfer.get("toUserAccount") or transfer.get("toTokenAccount")
                seller = seller or transfer.get("fromUserAccount") or transfer.get("fromTokenAccount")
        for transfer in native_transfers:
            sol_amount += abs(safe_float(transfer.get("amount"), 0.0)) / 1_000_000_000.0
            buyer = buyer or transfer.get("toUserAccount")
            seller = seller or transfer.get("fromUserAccount")
        fee = safe_float(tx.get("fee"), 0.0) / 1_000_000_000.0
        normalized.append({
            "mint": mint_row["mint"],
            "deployer": mint_row["deployer"],
            "graduation_timestamp": mint_row["graduation_timestamp"],
            "timestamp": ts,
            "signature": tx.get("signature"),
            "buyer": buyer,
            "seller": seller,
            "sol_amount": sol_amount,
            "token_amount": token_amount,
            "fee_sol": fee,
            "lpBurnPct": safe_float(mint_row.get("lpBurnPct", 0.0), 0.0),
            "lpLockedPct": safe_float(mint_row.get("lpLockedPct", 0.0), 0.0),
            "initial_liquidity_sol": safe_float(mint_row.get("initial_liquidity_sol", 0.0), 0.0),
            "topHolderPct": safe_float(mint_row.get("topHolderPct", 0.0), 0.0),
            "devHoldPct": safe_float(mint_row.get("devHoldPct", 0.0), 0.0),
            "mutableMetadata": bool(mint_row.get("mutableMetadata", False)),
            "mintAuthorityRenounced": bool(mint_row.get("mintAuthorityRenounced", True)),
            "freezeAuthorityRenounced": bool(mint_row.get("freezeAuthorityRenounced", True)),
            "transferTaxPct": safe_float(mint_row.get("transferTaxPct", 0.0), 0.0),
            "rugPullRisk": safe_float(mint_row.get("rugPullRisk", 0.0), 0.0),
            "honeypotRisk": safe_float(mint_row.get("honeypotRisk", 0.0), 0.0),
            "dangerSignals": safe_float(mint_row.get("dangerSignals", 0.0), 0.0),
        })
    return normalized


def collect_raw_data(force_refresh: bool = False) -> pd.DataFrame:
    if force_refresh:
        print("force_refresh=True: Deleting existing raw data and checkpoint to ensure a full refresh.")
        if RAW_DATA_PATH.exists():
            RAW_DATA_PATH.unlink() # Delete the parquet file
            print(f"  Deleted {RAW_DATA_PATH}")
        if COLLECTION_CHECKPOINT_PATH.exists():
            COLLECTION_CHECKPOINT_PATH.unlink() # Delete the checkpoint file
            print(f"  Deleted {COLLECTION_CHECKPOINT_PATH}")

    # First, always ensure the graduation manifest is available.
    # The new get_mint_list_from_dexscreener will create/overwrite it.
    manifest = get_mint_list_from_dexscreener(target=CONFIG['TARGET_TOKENS'])
    print(f"Mint list ready: {len(manifest)} tokens to process")

    # Now handle raw_data caching
    if RAW_DATA_PATH.exists() and not force_refresh:
        print(f"loading cached raw data from {RAW_DATA_PATH}")
        return pd.read_parquet(RAW_DATA_PATH)

    # If we get here, either force_refresh is True (and files were deleted) or RAW_DATA_PATH didn't exist initially.
    # So, we should start with a fresh processed set and empty raw_rows.
    # The checkpoint will also be empty or non-existent in this case.
    checkpoint = load_collection_checkpoint() # This will now return default if no file exists
    processed = set(checkpoint.get("processed_mints", []))
    raw_rows: list[dict[str, Any]] = []

    pending = manifest[~manifest["mint"].isin(processed)].reset_index(drop=True)
    print({"manifest_tokens": len(manifest), "already_processed": len(processed), "pending": len(pending)})

    total_attempted = len(pending)
    helius_mints_with_swaps = 0
    helius_mints_without_swaps = 0
    helius_api_calls_made = 0

    for idx, row in tqdm(pending.iterrows(), total=len(pending), desc="Collecting swaps"):
        helius_api_calls_made += 1
        swaps = fetch_swap_transactions(row["mint"])
        if swaps:
            helius_mints_with_swaps += 1
        else:
            helius_mints_without_swaps += 1
        raw_rows.extend(normalize_swap_rows(row, swaps))
        processed.add(row["mint"])
        if len(processed) % CONFIG["CHECKPOINT_EVERY_TOKENS"] == 0:
            pd.DataFrame(raw_rows).to_parquet(RAW_DATA_PATH, index=False)
            save_collection_checkpoint({"processed_mints": sorted(processed), "raw_rows": len(raw_rows)})

    raw_df = pd.DataFrame(raw_rows)
    if raw_df.empty:
        print("No raw rows collected. Check API key, manifest coverage, and Helius quota.")

    raw_df.to_parquet(RAW_DATA_PATH, index=False)
    save_collection_checkpoint({"processed_mints": sorted(processed), "raw_rows": len(raw_df)})

    # Add collection summary prints
    print(f"\nCollection summary:")
    print(f"  Mints attempted:     {total_attempted}")
    print(f"  Helius mints with swaps: {helius_mints_with_swaps}")
    print(f"  Helius mints without swaps: {helius_mints_without_swaps}")
    print(f"  Helius API calls made: {helius_api_calls_made}")
    print(f"  Birdeye success:     0 (Not directly tracked by Helius-based collection)")
    print(f"  Birdeye empty:       0 (Not directly tracked by Helius-based collection)")
    print(f"  Rugcheck filtered:   0 (Not directly tracked by this collection stage)")
    print(f"  Valid records saved: {len(raw_df)}")
    print(f"  Helius credits used: ~{helius_api_calls_made * 110} (estimation per Helius docs)")

    print({"raw_rows": len(raw_df), "tokens": raw_df['mint'].nunique(), "saved_to": str(RAW_DATA_PATH)})
    return raw_df

raw_df = collect_raw_data(force_refresh=True)
raw_df.head()

force_refresh=True: Deleting existing raw data and checkpoint to ensure a full refresh.
Fetching mint list from DexScreener...
  Got 20 candidates from DexScreener search
  Got additional tokens from profiles endpoint
  Total unique mints: 45
  Saved to /content/drive/MyDrive/meme_bot_training/graduated_tokens.parquet
Mint list ready: 45 tokens to process
{'manifest_tokens': 45, 'already_processed': 0, 'pending': 45}


DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'
DEBUG: HELIUS_API_KEY in helius_get: '03529aae-9455-4da7-a583-a0d06bb4a1e8'


KeyboardInterrupt: 

### Checking for Missing Values and Duplicates in `raw_df`

In [18]:
# Check for missing values
print('Missing values per column:')
display(raw_df.isnull().sum())

# Check for duplicate rows
duplicate_rows_count = raw_df.duplicated().sum()
print(f'Number of duplicate rows: {duplicate_rows_count}')

if duplicate_rows_count > 0:
    print('Displaying first 5 duplicate rows:')
    display(raw_df[raw_df.duplicated(keep=False)].sort_values(by=list(raw_df.columns)).head())

# Handle missing values in 'buyer' and 'seller' by filling with 'unknown'
raw_df['buyer'] = raw_df['buyer'].fillna('unknown')
raw_df['seller'] = raw_df['seller'].fillna('unknown')
print("\nFilled missing values in 'buyer' and 'seller' columns.")

Missing values per column:


,0
mint,0
deployer,0
graduation_timestamp,0
timestamp,0
signature,0
buyer,0
seller,0
sol_amount,0
token_amount,0
fee_sol,0


Number of duplicate rows: 0

Filled missing values in 'buyer' and 'seller' columns.


In [19]:
print("Cell 3b: Ground truth enrichment")
print("confirmed_rugs.csv not available — skipping")
print("Primary labels will come from Helius price data")
print("Ground truth enrichment can be added later")
print("Continuing to Cell 4...")

Cell 3b: Ground truth enrichment
confirmed_rugs.csv not available — skipping
Primary labels will come from Helius price data
Ground truth enrichment can be added later
Continuing to Cell 4...


In [24]:
FEATURES_PATH = Path(CONFIG["FEATURES_PATH"])


def compute_price_sol(row: pd.Series) -> float:
    token_amount = safe_float(row.get("token_amount"), 0.0)
    sol_amount = safe_float(row.get("sol_amount"), 0.0)
    if token_amount <= 0 or sol_amount <= 0:
        return np.nan
    return sol_amount / token_amount


def build_sequence(token_df: pd.DataFrame, decision_ts: pd.Timestamp) -> list[list[float]]:
    bucket_minutes = CONFIG["TIME_BUCKET_MINUTES"]
    history = token_df[token_df["timestamp"] <= decision_ts].copy()
    if history.empty:
        return [[0.0, 0.0, 0.0, 1.0, 0.0, 0.0] for _ in range(CONFIG["SEQUENCE_LENGTH"])]
    history["bucket"] = history["timestamp"].dt.floor(f"{bucket_minutes}min")
    grouped = []
    prev_price = None
    for bucket, chunk in history.groupby("bucket", sort=True):
        buyers = set(chunk["buyer"].dropna().astype(str))
        sellers = set(chunk["seller"].dropna().astype(str))
        holders = len(buyers | sellers)
        liquidity = safe_float(chunk["initial_liquidity_sol"].iloc[0], 0.0)
        volume = chunk["sol_amount"].fillna(0).sum()
        buy_count = chunk["buyer"].notna().sum()
        sell_count = chunk["seller"].notna().sum()
        ratio = buy_count / max(sell_count, 1)
        prices = chunk["price_sol"].dropna()
        current_price = float(prices.iloc[-1]) if not prices.empty else prev_price or 0.0
        velocity = 0.0 if prev_price in (None, 0.0) else (current_price - prev_price) / max(prev_price, 1e-9)
        prev_price = current_price
        grouped.append([float(holders), float(liquidity), float(volume), float(ratio), float(velocity), float(len(chunk))])
    grouped = grouped[-CONFIG["SEQUENCE_LENGTH"]:]
    while len(grouped) < CONFIG["SEQUENCE_LENGTH"]:
        grouped.insert(0, [0.0, 0.0, 0.0, 1.0, 0.0, 0.0])
    return grouped


def compute_labels(token_df: pd.DataFrame, decision_ts: pd.Timestamp, entry_price: float, initial_liquidity_sol: float) -> dict[str, float]:
    horizon_end = decision_ts + pd.Timedelta(hours=CONFIG["LABEL_WINDOW_HOURS"])
    future = token_df[(token_df["timestamp"] >= decision_ts) & (token_df["timestamp"] <= horizon_end)].copy()
    if future.empty or not np.isfinite(entry_price) or entry_price <= 0:
        return {
            "rug_label": 0.0,
            "time_to_rug_hours": float(CONFIG["LABEL_WINDOW_HOURS"]),
            "max_drawdown_pct": 0.0,
            "pump_2x_label": 0.0,
        }

    future["price_sol"] = future["price_sol"].ffill().bfill()
    future = future[np.isfinite(future["price_sol"])]
    if future.empty:
        return {
            "rug_label": 0.0,
            "time_to_rug_hours": float(CONFIG["LABEL_WINDOW_HOURS"]),
            "max_drawdown_pct": 0.0,
            "pump_2x_label": 0.0,
        }

    future["drawdown_pct"] = (entry_price - future["price_sol"]) / entry_price
    max_drawdown_pct = clamp(float(future["drawdown_pct"].max()), 0.0, 1.0) * 100.0
    price_drop_mask = future["drawdown_pct"] >= CONFIG["PRICE_DROP_RUG_THRESHOLD"]
    liq_proxy = future["sol_amount"].rolling(12, min_periods=1).sum()
    liquidity_drop_mask = (initial_liquidity_sol - liq_proxy) / max(initial_liquidity_sol, 1e-9) >= CONFIG["LIQUIDITY_DROP_RUG_THRESHOLD"]
    rug_rows = future[price_drop_mask & liquidity_drop_mask]
    rug_label = 1.0 if not rug_rows.empty else 0.0
    if rug_label == 1.0:
        first_rug_ts = rug_rows["timestamp"].iloc[0]
        time_to_rug_hours = min((first_rug_ts - decision_ts).total_seconds() / 3600.0, CONFIG["LABEL_WINDOW_HOURS"])
    else:
        time_to_rug_hours = float(CONFIG["LABEL_WINDOW_HOURS"])

    pump_2x_hit = float((future["price_sol"] >= entry_price * 2.0).any())
    return {
        "rug_label": rug_label,
        "time_to_rug_hours": float(time_to_rug_hours),
        "max_drawdown_pct": float(max_drawdown_pct),
        "pump_2x_label": pump_2x_hit,
    }


def build_feature_rows(raw_df: pd.DataFrame, force_refresh: bool = False) -> pd.DataFrame:
    if FEATURES_PATH.exists() and not force_refresh:
        print(f"loading cached features from {FEATURES_PATH}")
        return pd.read_parquet(FEATURES_PATH)

    df = raw_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["graduation_timestamp"] = pd.to_datetime(df["graduation_timestamp"], unit='s', utc=True) # FIX APPLIED HERE
    df["price_sol"] = df.apply(compute_price_sol, axis=1)

    feature_rows: list[dict[str, Any]] = []

    debug = {
        'processed': 0,
        'no_transactions': 0,
        'no_price_data': 0,
        'missing_required_fields': 0,
        'sequence_too_short': 0,
        'passed': 0
    }

    for mint, token_df in tqdm(df.groupby("mint", sort=True), desc="Building features"):
        debug['processed'] += 1
        token_df = token_df.sort_values("timestamp").reset_index(drop=True)
        meta = token_df.iloc[0]

        # Convert graduation_timestamp to datetime and add entry delay
        initial_decision_ts = pd.to_datetime(meta["graduation_timestamp"], unit='s', utc=True) + pd.Timedelta(seconds=CONFIG["ENTRY_DELAY_SECONDS"])

        # Determine the actual decision_ts. It should not be earlier than the first transaction.
        if not token_df.empty:
            earliest_tx_ts = token_df["timestamp"].min()
            decision_ts = max(initial_decision_ts, earliest_tx_ts)
        else:
            # If token_df is empty, we cannot proceed with this token.
            debug['no_transactions'] += 1
            if debug['no_transactions'] <= 3:
                print(f"  SKIP no_transactions (no token_df data at all): {mint}")
            continue

        pre_decision = token_df[token_df["timestamp"] <= decision_ts].copy()

        if pre_decision.empty:
            # This case should ideally not happen if token_df was not empty and decision_ts was set correctly.
            debug['no_transactions'] += 1
            if debug['no_transactions'] <= 3:
                print(f"  SKIP no_transactions (pre_decision still empty, investigate): {mint}")
            continue

        entry_prices = pre_decision["price_sol"].dropna()
        if entry_prices.empty:
            debug['no_price_data'] += 1
            if debug['no_price_data'] <= 3:
                print(f"  SKIP no_price_data: {mint}")
            continue
        entry_price = float(entry_prices.iloc[-1])
        sequence = build_sequence(token_df, decision_ts)

        # Add a check for sequence length, if needed, though build_sequence pads by default
        # This might be `sequence_too_short` if a minimum number of valid historical data points are required
        # For now, build_sequence handles padding, so this counter might not be incremented by existing logic.
        # If a new explicit check is added to build_sequence (e.g., minimum non-padded elements), then this counter would be useful.

        labels = compute_labels(token_df, decision_ts, entry_price, safe_float(meta["initial_liquidity_sol"], 0.0))
        danger_signals = clamp(safe_float(meta.get("dangerSignals"), 0.0) / 4.0)
        lp_locked_pct = safe_float(meta.get("lpLockedPct"), 0.0)
        unique_buyers = pre_decision["buyer"].dropna().astype(str).nunique()
        volatility = float(pd.Series([step[4] for step in sequence]).abs().mean())
        feature_rows.append({
            "mint": mint,
            "deployer": str(meta["deployer"]),
            "decision_timestamp": decision_ts,
            "entry_price_sol": entry_price,
            "tabular": [
                clamp(safe_float(meta.get("rugPullRisk", 0.0))),
                clamp(safe_float(meta.get("honeypotRisk", 0.0))),
                clamp(1.0 - safe_float(meta.get("lpBurnPct", 1.0))),
                clamp(safe_float(meta.get("transferTaxPct", 0.0))),
                clamp(safe_float(meta.get("topHolderPct", 0.0))),
                clamp(safe_float(meta.get("devHoldPct", 0.0))),
                1.0 if bool(meta.get("mutableMetadata", False)) else 0.0,
                0.0 if bool(meta.get("mintAuthorityRenounced", True)) else 1.0,
                0.0 if bool(meta.get("freezeAuthorityRenounced", True)) else 1.0,
                clamp(volatility),
                clamp(1.0 / max(safe_float(meta.get("initial_liquidity_sol", 0.05), 0.05), 0.05) / 5.0),
                clamp(1.0 - unique_buyers / 40.0),
                clamp(1.0 - lp_locked_pct / 100.0),
                danger_signals,
            ],
            "sequence": sequence,
            "rug_label": labels["rug_label"],
            "time_to_rug_hours": labels["time_to_rug_hours"],
            "max_drawdown_pct": labels["max_drawdown_pct"],
            "pump_2x_label": labels["pump_2x_label"],
        })
        debug['passed'] += 1

    features_df = pd.DataFrame(feature_rows)

    print("\nFeature build debug summary:")
    for k, v in debug.items():
        print(f"  {k}: {v}")

    if features_df.empty:
        print("\nAll tokens failed. Most common reason:")
        # Exclude 'processed' and 'passed' when finding the most common failure reason
        failure_reasons = {k: v for k, v in debug.items() if k not in ['processed', 'passed']}
        if failure_reasons:
            most_common = max(failure_reasons, key=failure_reasons.get)
            print(f"  {most_common}: {debug[most_common]}")
        else:
            print("  No specific failure reason identified beyond all tokens being skipped.")
        raise RuntimeError("No feature rows were created. Check graduation manifest coverage and Helius transaction availability.")
    features_df.to_parquet(FEATURES_PATH, index=False)
    print({"feature_rows": len(features_df), "tokens": features_df['mint'].nunique(), "saved_to": str(FEATURES_PATH)})
    return features_df

features_df = build_feature_rows(raw_df, force_refresh=False)
features_df.head()

Building features:   0%|          | 0/28 [00:00<?, ?it/s]

  SKIP no_price_data: BMqCw8whpusVnRtxZLrhi8Lm1LoZDkbLdDatUZjgpump
  SKIP no_price_data: pumpCmXqMfrsAkQ5r49WcJnRayYRqmXz6ae8H7H9Dfn

Feature build debug summary:
  processed: 28
  no_transactions: 0
  no_price_data: 2
  missing_required_fields: 0
  sequence_too_short: 0
  passed: 26
{'feature_rows': 26, 'tokens': 26, 'saved_to': '/content/drive/MyDrive/meme_bot_training/features.parquet'}


,mint,deployer,decision_timestamp,entry_price_sol,tabular,sequence,rug_label,time_to_rug_hours,max_drawdown_pct,pump_2x_label
0,226UbhEkT47hkJ3osqvi8PawtRsbHCykLiUD22rWpump,,2026-06-23 05:19:31+00:00,5.713524e-09,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0....",0.0,72.0,96.951498,1.0
1,4kFKrxf8xGd7nKGjy64Wg9whmZMyPspsBRfj2pe8pump,,2026-06-23 08:36:04+00:00,7.615196e-10,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0....",0.0,72.0,96.425641,1.0
2,4qucA2ru18HyMurKmUTVa1qSsPrrvDLvu3jo7Z9upump,,2026-06-23 07:33:42+00:00,3.109642e-07,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0....",0.0,72.0,99.647344,1.0
3,55rJv8SKoENoZF4t31hx98vRAkGt1LfVQNbUCw5Qpump,,2026-06-23 08:45:51+00:00,1.126412e-09,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0....",0.0,72.0,98.207630,1.0
4,58gCdwTpk8XrpVmr8uav7t4d2CUnWokdwP1QwhTKpump,,2026-06-23 08:50:17+00:00,4.204234e-07,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0....",0.0,72.0,99.951852,1.0


In [27]:
TRAIN_START = pd.Timestamp(CONFIG["TRAIN_START"])
TRAIN_END = pd.Timestamp(CONFIG["TRAIN_END"])
VAL_START = pd.Timestamp(CONFIG["VAL_START"])
VAL_END = pd.Timestamp(CONFIG["VAL_END"])
TEST_START = pd.Timestamp(CONFIG["TEST_START"])
TEST_END = pd.Timestamp(CONFIG["TEST_END"])


def split_temporal(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ts = pd.to_datetime(df["decision_timestamp"], utc=True)
    train_df = df[(ts >= TRAIN_START) & (ts <= TRAIN_END)].copy()
    val_df = df[(ts >= VAL_START) & (ts <= VAL_END)].copy()
    test_df = df[(ts >= TEST_START) & (ts <= TEST_END)].copy()
    if train_df.empty or val_df.empty or test_df.empty:
        raise RuntimeError({"train": len(train_df), "val": len(val_df), "test": len(test_df)})
    return train_df.sort_values("decision_timestamp"), val_df.sort_values("decision_timestamp"), test_df.sort_values("decision_timestamp")


def build_deployer_vocab(train_df: pd.DataFrame) -> dict[str, int]:
    deployers = sorted(train_df["deployer"].astype(str).unique())
    vocab = {"__UNK__": CONFIG["UNKNOWN_DEPLOYER_ID"]}
    vocab.update({deployer: idx + 1 for idx, deployer in enumerate(deployers)})
    return vocab


def attach_deployer_ids(df: pd.DataFrame, vocab: dict[str, int]) -> pd.DataFrame:
    out = df.copy()
    out["deployer_id"] = out["deployer"].astype(str).map(lambda value: vocab.get(value, CONFIG["UNKNOWN_DEPLOYER_ID"]))
    return out


def class_balance(df: pd.DataFrame) -> dict[str, float]:
    positives = float(df["rug_label"].sum())
    negatives = float(len(df) - positives)
    ratio = (negatives / max(positives, 1.0)) if positives > 0 else float("inf")
    return {
        "count": float(len(df)),
        "rugs": positives,
        "non_rugs": negatives,
        "non_rug_to_rug_ratio": ratio,
    }


# ── Temporal split with random fallback ──
try:
    train_df, val_df, test_df = split_temporal(features_df)
    print("Using temporal split")
except (RuntimeError, Exception) as e:
    print(f"Temporal split failed: {e}")
    print("Falling back to random stratified split (80/10/10)")
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(features_df))
    y_all = features_df["rug_label"].to_numpy()
    train_idx, temp_idx = train_test_split(idx, test_size=0.2, stratify=y_all, random_state=CONFIG["SEED"])
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y_all[temp_idx], random_state=CONFIG["SEED"])
    train_df = features_df.iloc[train_idx].copy().sort_values("decision_timestamp")
    val_df = features_df.iloc[val_idx].copy().sort_values("decision_timestamp")
    test_df = features_df.iloc[test_idx].copy().sort_values("decision_timestamp")
    print(f"Random split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

deployer_vocab = build_deployer_vocab(train_df)
train_df = attach_deployer_ids(train_df, deployer_vocab)
val_df = attach_deployer_ids(val_df, deployer_vocab)
test_df = attach_deployer_ids(test_df, deployer_vocab)

balance_report = {
    "train": class_balance(train_df),
    "validation": class_balance(val_df),
    "test": class_balance(test_df),
    "vocab_size_train_only": len(deployer_vocab),
    "unknown_in_val": int((val_df["deployer_id"] == 0).sum()),
    "unknown_in_test": int((test_df["deployer_id"] == 0).sum()),
}
print(json.dumps(balance_report, indent=2))

train_ratio = balance_report["train"]["non_rug_to_rug_ratio"]
if train_ratio > 10:
    print(f"Class imbalance exceeds 10:1. Suggested positive class weight ~= {train_ratio:.2f}")

# ── Class balance check: warn if any split is single-class ──
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    y = df["rug_label"].to_numpy()
    unique_classes = np.unique(y)
    rug_pct = y.mean() * 100
    print(f"{split_name}: {len(y)} samples, rug={rug_pct:.1f}%, classes={unique_classes}")
    if len(unique_classes) < 2:
        print(f"  WARNING: {split_name} set has only one class: {unique_classes}")
        print(f"  Model will need more data to learn both classes reliably.")

{
  "train": {
    "count": 4.0,
    "rugs": 0.0,
    "non_rugs": 4.0,
    "non_rug_to_rug_ratio": Infinity
  },
  "validation": {
    "count": 8.0,
    "rugs": 0.0,
    "non_rugs": 8.0,
    "non_rug_to_rug_ratio": Infinity
  },
  "test": {
    "count": 13.0,
    "rugs": 0.0,
    "non_rugs": 13.0,
    "non_rug_to_rug_ratio": Infinity
  },
  "vocab_size_train_only": 2,
  "unknown_in_val": 0,
  "unknown_in_test": 0
}
Class imbalance exceeds 10:1. Suggested positive class weight ~= inf


In [29]:
class RugDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.frame.iloc[idx]
        return {
            "tabular": torch.tensor(row["tabular"], dtype=torch.float32),
            "sequence": torch.tensor(row["sequence"], dtype=torch.float32),
            "deployer_id": torch.tensor(row["deployer_id"], dtype=torch.long),
            "rug_label": torch.tensor(row["rug_label"], dtype=torch.float32),
            "time_to_rug_hours": torch.tensor(min(row["time_to_rug_hours"], 72.0), dtype=torch.float32),
            "max_drawdown_pct": torch.tensor(row["max_drawdown_pct"], dtype=torch.float32),
            "pump_2x_label": torch.tensor(row["pump_2x_label"], dtype=torch.float32),
        }


def collate_batch(items: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:
    return {key: torch.stack([item[key] for item in items]) for key in items[0]}


def build_sampler(frame: pd.DataFrame):
    counts = frame["rug_label"].value_counts().to_dict()
    pos = counts.get(1.0, 0)
    neg = counts.get(0.0, 0)
    ratio = neg / max(pos, 1)
    if ratio <= 5:
        return None
    pos_weight = ratio
    weights = np.where(frame["rug_label"].to_numpy() >= 0.5, pos_weight, 1.0)
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)

train_dataset = RugDataset(train_df)
val_dataset = RugDataset(val_df)
test_dataset = RugDataset(test_df)
train_sampler = build_sampler(train_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    sampler=train_sampler,
    shuffle=train_sampler is None,
    collate_fn=collate_batch,
)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_batch)

print({
    "train_batches": len(train_loader),
    "val_batches": len(val_loader),
    "test_batches": len(test_loader),
    "weighted_sampler": train_sampler is not None,
})

{'train_batches': 1, 'val_batches': 1, 'test_batches': 1, 'weighted_sampler': False}


In [30]:
# Input contract for parity with TS inference/export:
# - tabular: [batch, 14]
# - sequence: [batch, 16, 6]
# - deployer_id: [batch]
# Outputs:
# - rug_prob: [batch, 1]
# - time_to_rug_hours: [batch, 1]
# - max_drawdown_pct: [batch, 1]
# - pump_2x_prob: [batch, 1]

class MCDropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p
        self.force_dropout = False

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return F.dropout(value, p=self.p, training=self.training or self.force_dropout)


class SequenceEncoder(nn.Module):
    def __init__(self, input_dim: int = 6, hidden_dim: int = 16, num_layers: int = 2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        if sequence.ndim != 3:
            raise ValueError("sequence tensor must be [batch, 16, 6]")
        _, hidden = self.gru(sequence)
        return hidden[-1]


class RugRiskNet(nn.Module):
    def __init__(
        self,
        tabular_dim: int = 14,
        num_deployers: int = 1,
        deployer_dim: int = 32,
        sequence_dim: int = 16,
        feature_mean: np.ndarray | None = None,
        feature_std: np.ndarray | None = None,
    ):
        super().__init__()
        input_dim = tabular_dim + deployer_dim + sequence_dim
        self.register_buffer("feature_mean", torch.tensor(feature_mean if feature_mean is not None else np.zeros(tabular_dim), dtype=torch.float32))
        self.register_buffer("feature_std", torch.tensor(feature_std if feature_std is not None else np.ones(tabular_dim), dtype=torch.float32))
        self.deployer_embedding = nn.Embedding(num_deployers, deployer_dim)
        nn.init.normal_(self.deployer_embedding.weight, mean=0.0, std=0.02)
        self.sequence_encoder = SequenceEncoder(input_dim=6, hidden_dim=sequence_dim, num_layers=2)

        self.input_skip = nn.Linear(input_dim, 128)
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            MCDropout(CONFIG["DROPOUT_1"]),
        )
        self.skip2 = nn.Linear(128, 64)
        self.block2 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            MCDropout(CONFIG["DROPOUT_2"]),
        )
        self.block3 = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.rug_head = nn.Linear(32, 1)
        self.time_head = nn.Linear(32, 1)
        self.drawdown_head = nn.Linear(32, 1)
        self.pump_head = nn.Linear(32, 1)

    def encode_backbone(self, tabular: torch.Tensor, deployer_id: torch.Tensor, sequence: torch.Tensor) -> torch.Tensor:
        tabular = (tabular - self.feature_mean) / torch.clamp(self.feature_std, min=1e-6)
        deployer = self.deployer_embedding(deployer_id.long())
        temporal = self.sequence_encoder(sequence)
        x = torch.cat([tabular, deployer, temporal], dim=1)
        h1 = self.block1(x) + self.input_skip(x)
        h2 = self.block2(h1) + self.skip2(h1)
        return self.block3(h2)

    def forward_logits(self, tabular: torch.Tensor, sequence: torch.Tensor, deployer_id: torch.Tensor):
        z = self.encode_backbone(tabular, deployer_id, sequence)
        rug_logits = self.rug_head(z)
        time_to_rug = torch.clamp(F.relu(self.time_head(z)), max=72.0)
        max_drawdown = torch.clamp(100.0 * torch.sigmoid(self.drawdown_head(z)), min=0.0, max=100.0)
        pump_logits = self.pump_head(z)
        return rug_logits, time_to_rug, max_drawdown, pump_logits

    def forward(self, tabular: torch.Tensor, sequence: torch.Tensor, deployer_id: torch.Tensor):
        rug_logits, time_to_rug, max_drawdown, pump_logits = self.forward_logits(tabular, sequence, deployer_id)
        return torch.sigmoid(rug_logits), time_to_rug, max_drawdown, torch.sigmoid(pump_logits)

feature_mean = np.asarray(train_df["tabular"].tolist(), dtype=np.float32).mean(axis=0)
feature_std = np.clip(np.asarray(train_df["tabular"].tolist(), dtype=np.float32).std(axis=0), 1e-4, None)
model = RugRiskNet(num_deployers=len(deployer_vocab), feature_mean=feature_mean, feature_std=feature_std).to(DEVICE)
print(model)
# XGBoost classifier for tabular features
xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=CONFIG['XGB_N_ESTIMATORS'],
    max_depth=CONFIG['XGB_MAX_DEPTH'],
    learning_rate=CONFIG['XGB_LEARNING_RATE'],
    subsample=CONFIG['XGB_SUBSAMPLE'],
    colsample_bytree=CONFIG['XGB_COLSAMPLE'],
    use_label_encoder=False,
    eval_metric='auc',
    tree_method='hist',
    device='cuda'
)

TABULAR_FEATURE_NAMES = [
    'rugPullRisk', 'honeypotRisk', 'lpBurnGap',
    'transferTaxPct', 'topHolderPct', 'devHoldPct',
    'mutableMetadata', 'mintAuthority', 'freezeAuthority',
    'volatility1m', 'lowLiquidity', 'lowBuyers',
    'rugcheckLpUnlocked', 'rugcheckDangerSignals'
]


RugRiskNet(
  (deployer_embedding): Embedding(2, 32)
  (sequence_encoder): SequenceEncoder(
    (gru): GRU(6, 16, num_layers=2, batch_first=True)
  )
  (input_skip): Linear(in_features=62, out_features=128, bias=True)
  (block1): Sequential(
    (0): Linear(in_features=62, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MCDropout()
  )
  (skip2): Linear(in_features=128, out_features=64, bias=True)
  (block2): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MCDropout()
  )
  (block3): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
  )
  (rug_head): Linear(in_features=32, out_features=1, bias=True)
  (time_head): Linear(in_features=32, out_features=1, bias=True)
  (drawdown_head): Linear(in_features=32, out_features=

In [38]:
# Prepare tabular data for XGBoost
X_train_tabular = np.asarray(train_df["tabular"].tolist(), dtype=np.float32)
y_train_rug = train_df["rug_label"].to_numpy(dtype=np.float32)
X_val_tabular = np.asarray(val_df["tabular"].tolist(), dtype=np.float32)
y_val_rug = val_df["rug_label"].to_numpy(dtype=np.float32)

# Add dataset size check at the start of training
MIN_TOKENS = 500
if len(features_df) < MIN_TOKENS:
    raise ValueError(
        f"Not enough data to train: {len(features_df)} tokens. "
        f"Need at least {MIN_TOKENS}. "
        f"Re-run Cell 3 collection."
    )

print(f"Dataset size check: {len(features_df)} tokens ✓")
print(f"Class balance:")
print(features_df['rug_label'].value_counts())

# Train XGBoost first (seconds)
print("Training XGBoost...")
xgb_clf.fit(
    X_train_tabular, y_train_rug,
    eval_set=[(X_val_tabular, y_val_rug)],
    verbose=100
)

# Feature importance
importances = dict(zip(
    TABULAR_FEATURE_NAMES,
    xgb_clf.feature_importances_
))
print("\nTop 5 features by importance:")
for feat, imp in sorted(
    importances.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]:
    print(f"  {feat}: {imp:.4f}")

# PyTorch RugRiskNet (20-40 mins on T4)
print("\nTraining PyTorch RugRiskNet...")

def move_batch(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {key: value.to(DEVICE) for key, value in batch.items()}


def compute_positive_class_weight(frame: pd.DataFrame) -> float:
    pos = float(frame["rug_label"].sum())
    neg = float(len(frame) - pos)
    return max(1.0, neg / max(pos, 1.0))


RUG_POS_WEIGHT = compute_positive_class_weight(train_df)
RUG_BCE = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([RUG_POS_WEIGHT], device=DEVICE))
PUMP_BCE = nn.BCEWithLogitsLoss()
TIME_MSE = nn.MSELoss(reduction="none")
DD_MSE = nn.MSELoss()


def multitask_loss(model: RugRiskNet, batch: dict[str, torch.Tensor]):
    rug_logits, time_pred, drawdown_pred, pump_logits = model.forward_logits(batch["tabular"], batch["sequence"], batch["deployer_id"])
    rug_loss = RUG_BCE(rug_logits.view(-1), batch["rug_label"].view(-1))
    rug_mask = batch["rug_label"].view(-1) > 0.5
    if rug_mask.any():
        time_loss = TIME_MSE(time_pred.view(-1)[rug_mask], batch["time_to_rug_hours"].view(-1)[rug_mask]).mean()
    else:
        time_loss = torch.tensor(0.0, device=DEVICE)
    drawdown_loss = DD_MSE(drawdown_pred.view(-1), batch["max_drawdown_pct"].view(-1))
    pump_loss = PUMP_BCE(pump_logits.view(-1), batch["pump_2x_label"].view(-1))
    total = 0.5 * rug_loss + 0.2 * time_loss + 0.2 * drawdown_loss + 0.1 * pump_loss
    return total, {
        "rug": float(rug_loss.detach().cpu()),
        "time": float(time_loss.detach().cpu()),
        "drawdown": float(drawdown_loss.detach().cpu()),
        "pump": float(pump_loss.detach().cpu()),
    }


@torch.no_grad()
def evaluate_epoch(model: RugRiskNet, loader: DataLoader):
    model.eval()
    losses = []
    rug_probs = []
    rug_labels = []
    for batch in loader:
        batch = move_batch(batch)
        loss, _ = multitask_loss(model, batch)
        rug_prob, _, _, _ = model(batch["tabular"], batch["sequence"], batch["deployer_id"])
        losses.append(float(loss.detach().cpu()))
        rug_probs.extend(rug_prob.view(-1).detach().cpu().numpy().tolist())
        rug_labels.extend(batch["rug_label"].view(-1).detach().cpu().numpy().tolist())
    val_auc = roc_auc_score(rug_labels, rug_probs) if len(set(rug_labels)) > 1 else 0.5
    return {"loss": float(np.mean(losses)), "rug_auc": float(val_auc)}


optimizer = torch.optim.AdamW(
    [
        {"params": [param for name, param in model.named_parameters() if not name.startswith("deployer_embedding")], "lr": CONFIG["LEARNING_RATE"]},
        {"params": model.deployer_embedding.parameters(), "lr": CONFIG["LEARNING_RATE"] * 0.1},
    ],
    weight_decay=CONFIG["WEIGHT_DECAY"],
)

history = []
best_val_auc = -1.0
best_epoch = -1
patience_left = CONFIG["PATIENCE"]

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    model.train()
    train_losses = []
    for batch in train_loader:
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)
        loss, _ = multitask_loss(model, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        train_losses.append(float(loss.detach().cpu()))

    val_metrics = evaluate_epoch(model, val_loader)
    train_loss = float(np.mean(train_losses)) if train_losses else 0.0
    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["rug_auc"],
    }
    history.append(record)
    print(record)

    if val_metrics["rug_auc"] > best_val_auc:
        best_val_auc = val_metrics["rug_auc"]
        best_epoch = epoch
        patience_left = CONFIG["PATIENCE"]
        torch.save(
            {
                "epoch": epoch,
                "state_dict": model.state_dict(),
                "feature_mean": feature_mean,
                "feature_std": feature_std,
                "deployer_vocab": deployer_vocab,
                "history": history,
                "best_val_auc": best_val_auc,
            },
            CONFIG["BEST_CHECKPOINT_PATH"],
        )
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch}; best epoch={best_epoch}, best val AUC={best_val_auc:.4f}")
            break

print({"best_epoch": best_epoch, "best_val_auc": best_val_auc, "checkpoint": CONFIG["BEST_CHECKPOINT_PATH"]})

ValueError: Not enough data to train: 26 tokens. Need at least 500. Re-run Cell 3 collection.

In [39]:
from numpy import unique

checkpoint = torch.load(CONFIG["BEST_CHECKPOINT_PATH"], map_location=DEVICE, weights_only=False)
model = RugRiskNet(
    num_deployers=len(checkpoint["deployer_vocab"]),
    feature_mean=np.asarray(checkpoint["feature_mean"], dtype=np.float32),
    feature_std=np.asarray(checkpoint["feature_std"], dtype=np.float32),
).to(DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

@torch.no_grad()
def collect_predictions(model: RugRiskNet, loader: DataLoader):
    out = {"rug": [], "time": [], "drawdown": [], "pump": [], "rug_y": [], "time_y": [], "drawdown_y": [], "pump_y": []}
    for batch in loader:
        batch = move_batch(batch)
        rug_prob, time_pred, drawdown_pred, pump_prob = model(batch["tabular"], batch["sequence"], batch["deployer_id"])
        out["rug"].extend(rug_prob.view(-1).detach().cpu().numpy().tolist())
        out["time"].extend(time_pred.view(-1).detach().cpu().numpy().tolist())
        out["drawdown"].extend(drawdown_pred.view(-1).detach().cpu().numpy().tolist())
        out["pump"].extend(pump_prob.view(-1).detach().cpu().numpy().tolist())
        out["rug_y"].extend(batch["rug_label"].view(-1).detach().cpu().numpy().tolist())
        out["time_y"].extend(batch["time_to_rug_hours"].view(-1).detach().cpu().numpy().tolist())
        out["drawdown_y"].extend(batch["max_drawdown_pct"].view(-1).detach().cpu().numpy().tolist())
        out["pump_y"].extend(batch["pump_2x_label"].view(-1).detach().cpu().numpy().tolist())
    return out

preds = collect_predictions(model, test_loader)

# --- FIX 1: Guard against single-class test set. ---
# Convert y_test_rug to numpy array before checking unique values
y_test_rug_np = np.asarray(preds["rug_y"], dtype=np.float32)

if len(unique(y_test_rug_np)) < 2:
    print("WARNING: Test set has only one class")
    print(f"  y_test_rug unique values: {unique(y_test_rug_np)}")
    print(f"  Test set size: {len(y_test_rug_np)}")
    print(f"  Train set size: {len(y_train_rug)}")
    print(f"  Total dataset size: {len(features_df)}")
    print("")
    print("ROOT CAUSE: Dataset is too small for temporal split")
    print("  Need at least 500+ tokens for meaningful split")
    print("  Currently have too few tokens with valid features")
    print("")
    print("NEXT STEP: Re-run Cell 3 to collect more data")
    print("  Target: 5000+ tokens minimum")
    print("  Current: insufficient")
    raise ValueError(
        "Dataset too small for training. "
        "Re-run Cell 3 to collect more data. "
        f"Current valid tokens: {len(features_df)}"
    )

rug_auc = roc_auc_score(preds["rug_y"], preds["rug"]) if len(set(preds["rug_y"])) > 1 else 0.5
pump_auc = roc_auc_score(preds["pump_y"], preds["pump"]) if len(set(preds["pump_y"])) > 1 else 0.5
rug_hard = [1 if value >= 0.5 else 0 for value in preds["rug"]]
mask_rug = np.asarray(preds["rug_y"]) >= 0.5
mae_time = float(np.mean(np.abs(np.asarray(preds["time"])[mask_rug] - np.asarray(preds["time_y"])[mask_rug]))) if mask_rug.any() else 0.0
mae_drawdown = float(np.mean(np.abs(np.asarray(preds["drawdown"]) - np.asarray(preds["drawdown_y"]))))


# ── Hybrid ensemble evaluation ──
X_test_tabular = np.asarray(test_df["tabular"].tolist(), dtype=np.float32)
y_test_rug = test_df["rug_label"].to_numpy(dtype=np.float32)
xgb_probs = xgb_clf.predict_proba(X_test_tabular)[:, 1]
xgb_auc = roc_auc_score(y_test_rug, xgb_probs) if len(set(y_test_rug)) > 1 else 0.5
torch_probs = np.asarray(preds["rug"])
torch_auc = roc_auc_score(preds["rug_y"], torch_probs) if len(set(preds["rug_y"])) > 1 else 0.5
ensemble_probs = (
    xgb_probs * CONFIG["XGB_ENSEMBLE_WEIGHT"] +
    torch_probs * CONFIG["TORCH_ENSEMBLE_WEIGHT"]
)
ensemble_auc = roc_auc_score(y_test_rug, ensemble_probs) if len(set(y_test_rug)) > 1 else 0.5

print("=" * 40)
print(f"XGBoost AUC:  {xgb_auc:.4f}")
print(f"PyTorch AUC:  {torch_auc:.4f}")
print(f"Ensemble AUC: {ensemble_auc:.4f} <-- quality gate")
print("=" * 40)

# --- FIX 2: Fix classification_report to handle single class gracefully: ---
preds_binary = (ensemble_probs >= 0.5).astype(int)
print(classification_report(
    y_test_rug,
    preds_binary,
    target_names=["not_rug", "rug"],
    labels=[0, 1],          # explicitly specify both labels
    zero_division=0         # handle missing class gracefully
))

model_meta = {
    "version": 1,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "architecture": "tabular_residual_mlp_sequence_gru_deployer_embedding_multitask",
    "onnx_opset": CONFIG["EXPORT_OPSET"],
    "feature_names": CONFIG["TABULAR_FEATURES"],
    "temporal_split": {
        "train": f"{CONFIG['TRAIN_START']} to {CONFIG['TRAIN_END']}",
        "validation": f"{CONFIG['VAL_START']} to {CONFIG['VAL_END']}",
        "test": f"{CONFIG['TEST_START']} to {CONFIG['TEST_END']}",
    },
    "metrics": {
        "history": checkpoint["history"],
        "test": {
            "rug_auc": float(rug_auc),
            "xgb_auc": float(xgb_auc),
            "torch_auc": float(torch_auc),
            "ensemble_auc": float(ensemble_auc),
            "precision": float(precision_score(preds["rug_y"], rug_hard, zero_division=0)),
            "recall": float(recall_score(preds["rug_y"], rug_hard, zero_division=0)),
            "f1": float(f1_score(preds["rug_y"], rug_hard, zero_division=0)),
            "time_to_rug_mae": mae_time,
            "max_drawdown_mae": mae_drawdown,
            "pump_2x_auc": float(pump_auc),
        },
        "validation_best_auc": float(checkpoint["best_val_auc"]),
    },
}

print(json.dumps(model_meta, indent=2))
QUALITY_GATE_PASSED = ensemble_auc >= CONFIG["AUC_QUALITY_GATE"]
if not QUALITY_GATE_PASSED:
    print("MODEL FAILED QUALITY GATE — DO NOT EXPORT")
else:
    print("Quality gate passed; Cell 10 export is enabled.")

  y_test_rug unique values: [0.]
  Test set size: 13
  Train set size: 4
  Total dataset size: 26

ROOT CAUSE: Dataset is too small for temporal split
  Need at least 500+ tokens for meaningful split
  Currently have too few tokens with valid features

NEXT STEP: Re-run Cell 3 to collect more data
  Target: 5000+ tokens minimum
  Current: insufficient


ValueError: Dataset too small for training. Re-run Cell 3 to collect more data. Current valid tokens: 26

In [37]:
RAW_DATA_PATH = Path(CONFIG["RAW_DATA_PATH"])
COLLECTION_CHECKPOINT_PATH = Path(CONFIG["COLLECTION_CHECKPOINT_PATH"])


def clamp(value: float, low: float = 0.0, high: float = 1.0) -> float:
    return max(low, min(high, value))


def safe_float(value: Any, fallback: float = 0.0) -> float:
    try:
        if value is None or value == "":
            return fallback
        value = float(value)
        return value if math.isfinite(value) else fallback
    except Exception:
        return fallback


def to_timestamp(value: Any) -> pd.Timestamp:
    if isinstance(value, pd.Timestamp):
        return value.tz_convert("UTC") if value.tzinfo else value.tz_localize("UTC")
    if isinstance(value, (int, float)):
        unit = "ms" if value > 10_000_000_000 else "s"
        return pd.to_datetime(value, unit=unit, utc=True)
    return pd.to_datetime(value, utc=True)


def with_backoff(fn, *args, **kwargs):
    delay = CONFIG["INITIAL_BACKOFF_SECONDS"]
    last_error = None
    for attempt in range(CONFIG["MAX_RETRIES"]):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            last_error = exc
            if attempt == CONFIG["MAX_RETRIES"] - 1:
                break
            sleep_for = delay * (2 ** attempt)
            print(f"retry {attempt + 1}/{CONFIG['MAX_RETRIES']} after error: {exc}")
            time.sleep(sleep_for)
    raise last_error


def helius_get(path: str, params: dict[str, Any]) -> Any:
    url = f"{CONFIG['HELIUS_BASE_URL'].rstrip('/')}{path}"
    # Debug print: Check the API key value just before it's used
    print(f"DEBUG: HELIUS_API_KEY in helius_get: '{CONFIG['HELIUS_API_KEY']}'")
    params = {**params, "api-key": CONFIG["HELIUS_API_KEY"]}
    response = requests.get(url, params=params, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"])
    if response.status_code in {429, 500, 502, 503, 504}:
        raise RuntimeError(f"helius_transient_http_{response.status_code}")
    response.raise_for_status()
    return response.json()

def get_mint_list_from_dexscreener(target=500):
    """
    Fetches recently graduated Solana tokens from DexScreener.
    Free API, no key required.
    Returns a DataFrame with columns:
    mint, graduation_timestamp, initial_liquidity_sol
    """
    print("Fetching mint list from DexScreener...")
    mints = []

    # Search for Raydium pairs on Solana created after Jan 2025
    url = "https://api.dexscreener.com/latest/dex/search"
    params = {'q': 'pump', 'chainId': 'solana'}

    try:
        r = requests.get(url, params=params, timeout=15)
        pairs = r.json().get('pairs', [])

        start_ts = int(datetime(2025, 1, 1).timestamp() * 1000)

        for p in pairs:
            if (p.get('dexId') == 'raydium' and
                p.get('pairCreatedAt', 0) >= start_ts and
                p.get('baseToken', {}).get('address') and
                p.get('liquidity', {}).get('usd', 0) > 1000):

                mints.append({
                    'mint': p['baseToken']['address'],
                    'graduation_timestamp': p['pairCreatedAt'] / 1000,
                    'initial_liquidity_sol': p.get(
                        'liquidity', {}).get('usd', 0) / 150,
                    'deployer': ''
                })

        print(f"  Got {len(mints)} candidates from DexScreener search")
    except Exception as e:
        print(f"  DexScreener search error: {e}")

    # Also try token profiles endpoint
    try:
        profiles_url = "https://api.dexscreener.com/token-profiles/latest/v1"
        r2 = requests.get(profiles_url, timeout=15)
        profiles = r2.json() if r2.status_code == 200 else []

        if isinstance(profiles, list):
            for p in profiles:
                if p.get('chainId') == 'solana':
                    mints.append({
                        'mint': p.get('tokenAddress', ''),
                        'graduation_timestamp': time.time() - 86400,
                        'initial_liquidity_sol': 0,
                        'deployer': ''
                    })
        print(f"  Got additional tokens from profiles endpoint")
    except Exception as e:
        print(f"  Profiles endpoint error: {e}")

    # Deduplicate
    seen = set()
    unique = []
    for m in mints:
        if m['mint'] and m['mint'] not in seen:
            seen.add(m['mint'])
            unique.append(m)

    print(f"  Total unique mints: {len(unique)}")

    # Save as parquet so Cell 3 can load it
    df = pd.DataFrame(unique[:target * 3])
    save_path = Path(CONFIG['GRADUATION_MANIFEST_PATH'])
    save_path.parent.mkdir(parents=True, exist_ok=True) # Ensure directory exists
    df.to_parquet(save_path, index=False)
    print(f"  Saved to {save_path}")
    return df


def load_collection_checkpoint() -> dict[str, Any]:
    if COLLECTION_CHECKPOINT_PATH.exists():
        return json.loads(COLLECTION_CHECKPOINT_PATH.read_text())
    return {"processed_mints": [], "raw_rows": 0, "updated_at": None}


def save_collection_checkpoint(state: dict[str, Any]) -> None:
    state["updated_at"] = datetime.now(timezone.utc).isoformat()
    COLLECTION_CHECKPOINT_PATH.write_text(json.dumps(state, indent=2))


def fetch_swap_transactions(mint: str) -> list[dict[str, Any]]:
    before = None
    rows: list[dict[str, Any]] = []
    for _ in range(CONFIG["MAX_SWAP_PAGES"]):
        params = {"type": "SWAP", "limit": CONFIG["SWAP_PAGE_LIMIT"]}
        if before:
            params["before"] = before
        payload = with_backoff(helius_get, f"/v0/addresses/{mint}/transactions", params)
        if not payload:
            break
        rows.extend(payload)
        if len(payload) < CONFIG["SWAP_PAGE_LIMIT"]:
            break
        before = payload[-1].get("signature")
        if not before:
            break
    return rows


def normalize_swap_rows(mint_row: pd.Series, swaps: list[dict[str, Any]]) -> list[dict[str, Any]]:
    normalized: list[dict[str, Any]] = []
    for tx in swaps:
        ts = to_timestamp(tx.get("timestamp") or tx.get("blockTime") or tx.get("slotTime"))
        token_transfers = tx.get("tokenTransfers") or []
        native_transfers = tx.get("nativeTransfers") or []
        token_amount = 0.0
        sol_amount = 0.0
        buyer = None
        seller = None
        for transfer in token_transfers:
            if transfer.get("mint") == mint_row["mint"]:
                token_amount += abs(safe_float(transfer.get("tokenAmount") or transfer.get("rawTokenAmount", {}).get("tokenAmount")))
                buyer = buyer or transfer.get("toUserAccount") or transfer.get("toTokenAccount")
                seller = seller or transfer.get("fromUserAccount") or transfer.get("fromTokenAccount")
        for transfer in native_transfers:
            sol_amount += abs(safe_float(transfer.get("amount"), 0.0)) / 1_000_000_000.0
            buyer = buyer or transfer.get("toUserAccount")
            seller = seller or transfer.get("fromUserAccount")
        fee = safe_float(tx.get("fee"), 0.0) / 1_000_000_000.0
        normalized.append({
            "mint": mint_row["mint"],
            "deployer": mint_row["deployer"],
            "graduation_timestamp": mint_row["graduation_timestamp"],
            "timestamp": ts,
            "signature": tx.get("signature"),
            "buyer": buyer,
            "seller": seller,
            "sol_amount": sol_amount,
            "token_amount": token_amount,
            "fee_sol": fee,
            "lpBurnPct": safe_float(mint_row.get("lpBurnPct", 0.0), 0.0),
            "lpLockedPct": safe_float(mint_row.get("lpLockedPct", 0.0), 0.0),
            "initial_liquidity_sol": safe_float(mint_row.get("initial_liquidity_sol", 0.0), 0.0),
            "topHolderPct": safe_float(mint_row.get("topHolderPct", 0.0), 0.0),
            "devHoldPct": safe_float(mint_row.get("devHoldPct", 0.0), 0.0),
            "mutableMetadata": bool(mint_row.get("mutableMetadata", False)),
            "mintAuthorityRenounced": bool(mint_row.get("mintAuthorityRenounced", True)),
            "freezeAuthorityRenounced": bool(mint_row.get("freezeAuthorityRenounced", True)),
            "transferTaxPct": safe_float(mint_row.get("transferTaxPct", 0.0), 0.0),
            "rugPullRisk": safe_float(mint_row.get("rugPullRisk", 0.0), 0.0),
            "honeypotRisk": safe_float(mint_row.get("honeypotRisk", 0.0), 0.0),
            "dangerSignals": safe_float(mint_row.get("dangerSignals", 0.0), 0.0),
        })
    return normalized


def collect_raw_data(force_refresh: bool = False) -> pd.DataFrame:
    # First, always ensure the graduation manifest is available.
    # The new get_mint_list_from_dexscreener will create/overwrite it.
    manifest = get_mint_list_from_dexscreener(target=CONFIG['TARGET_TOKENS'])
    print(f"Mint list ready: {len(manifest)} tokens to process")

    # Now handle raw_data caching
    if RAW_DATA_PATH.exists() and not force_refresh:
        print(f"loading cached raw data from {RAW_DATA_PATH}")
        return pd.read_parquet(RAW_DATA_PATH)

    # If we get here, either force_refresh is True or RAW_DATA_PATH didn't exist.
    # We already have `manifest` from the previous step.
    checkpoint = load_collection_checkpoint()
    processed = set(checkpoint.get("processed_mints", []))
    raw_rows: list[dict[str, Any]] = []
    if RAW_DATA_PATH.exists() and processed:
        raw_rows.extend(pd.read_parquet(RAW_DATA_PATH).to_dict("records"))

    pending = manifest[~manifest["mint"].isin(processed)].reset_index(drop=True)
    print({"manifest_tokens": len(manifest), "already_processed": len(processed), "pending": len(pending)})

    total_attempted = len(pending)
    helius_mints_with_swaps = 0
    helius_mints_without_swaps = 0
    helius_api_calls_made = 0

    for idx, row in tqdm(pending.iterrows(), total=len(pending), desc="Collecting swaps"):
        helius_api_calls_made += 1
        swaps = fetch_swap_transactions(row["mint"])
        if swaps:
            helius_mints_with_swaps += 1
        else:
            helius_mints_without_swaps += 1
        raw_rows.extend(normalize_swap_rows(row, swaps))
        processed.add(row["mint"])
        if len(processed) % CONFIG["CHECKPOINT_EVERY_TOKENS"] == 0:
            pd.DataFrame(raw_rows).to_parquet(RAW_DATA_PATH, index=False)
            save_collection_checkpoint({"processed_mints": sorted(processed), "raw_rows": len(raw_rows)})

    raw_df = pd.DataFrame(raw_rows)
    if raw_df.empty:
        print("No raw rows collected. Check API key, manifest coverage, and Helius quota.")

    raw_df.to_parquet(RAW_DATA_PATH, index=False)
    save_collection_checkpoint({"processed_mints": sorted(processed), "raw_rows": len(raw_df)})

    # Add collection summary prints
    print(f"\nCollection summary:")
    print(f"  Mints attempted:     {total_attempted}")
    print(f"  Helius mints with swaps: {helius_mints_with_swaps}")
    print(f"  Helius mints without swaps: {helius_mints_without_swaps}")
    print(f"  Helius API calls made: {helius_api_calls_made}")
    print(f"  Birdeye success:     0 (Not directly tracked by Helius-based collection)")
    print(f"  Birdeye empty:       0 (Not directly tracked by Helius-based collection)")
    print(f"  Rugcheck filtered:   0 (Not directly tracked by this collection stage)")
    print(f"  Valid records saved: {len(raw_df)}")
    print(f"  Helius credits used: ~{helius_api_calls_made * 110} (estimation per Helius docs)")

    print({"raw_rows": len(raw_df), "tokens": raw_df['mint'].nunique(), "saved_to": str(RAW_DATA_PATH)})
    return raw_df

raw_df = collect_raw_data(force_refresh=False)
raw_df.head()

Fetching mint list from DexScreener...
  Got 3 candidates from DexScreener search
  Got additional tokens from profiles endpoint
  Total unique mints: 27
  Saved to /content/drive/MyDrive/meme_bot_training/graduated_tokens.parquet
Mint list ready: 27 tokens to process
loading cached raw data from /content/drive/MyDrive/meme_bot_training/raw_data.parquet


,mint,deployer,graduation_timestamp,timestamp,signature,buyer,seller,sol_amount,token_amount,fee_sol,...,initial_liquidity_sol,topHolderPct,devHoldPct,mutableMetadata,mintAuthorityRenounced,freezeAuthorityRenounced,transferTaxPct,rugPullRisk,honeypotRisk,dangerSignals
0,pumpCmXqMfrsAkQ5r49WcJnRayYRqmXz6ae8H7H9Dfn,,1.752513e+09,2026-06-23 08:51:02+00:00,4ggMEAi38QZCj7yhFrgZwvcvFF2YnzQ9HGqwPjm75C7DUU...,99mRw3EzdJZWEUjgp1nrU4WeHsukUBjbh7gYE7pm4F3c,8tpmZEtYup1UEXNqaKoXWE5eiK7yqxh9cGfh76YeCtuC,0.407568,20070.606028,0.000016,...,549.874733,0.0,0.0,False,True,True,0.0,0.0,0.0,0.0
1,pumpCmXqMfrsAkQ5r49WcJnRayYRqmXz6ae8H7H9Dfn,,1.752513e+09,2026-06-23 08:51:01+00:00,4bcczboZSdCMutfvcvomyHUUJaaonvd2kuJcYKRPCYk9bY...,MfDuWeqSHEqTFVYZ7LoexgAK9dxk7cy4DFJWjWMGVWa,BofA2ViUSudPBTUms2KRuG6AHNeMawjNfwqTJDgx5BKW,0.000000,50636.936361,0.000005,...,549.874733,0.0,0.0,False,True,True,0.0,0.0,0.0,0.0
2,pumpCmXqMfrsAkQ5r49WcJnRayYRqmXz6ae8H7H9Dfn,,1.752513e+09,2026-06-23 08:51:01+00:00,3QwqNPoQsGR3de7kG2wuvhyN9pYDaRY5WeZa3om5WdoKSU...,D5YqVMoSxnqeZAKAUUE1Dm3bmjtdxQ5DCF356ozqN9cM,8tpmZEtYup1UEXNqaKoXWE5eiK7yqxh9cGfh76YeCtuC,0.000000,117.893270,0.000005,...,549.874733,0.0,0.0,False,True,True,0.0,0.0,0.0,0.0
3,pumpCmXqMfrsAkQ5r49WcJnRayYRqmXz6ae8H7H9Dfn,,1.752513e+09,2026-06-23 08:50:59+00:00,2wrqDcXHQzfSzWuhqtYPgEAZncxpczAUpZWSBVKWdUNKDG...,D5YqVMoSxnqeZAKAUUE1Dm3bmjtdxQ5DCF356ozqN9cM,8tpmZEtYup1UEXNqaKoXWE5eiK7yqxh9cGfh76YeCtuC,0.000000,511.970414,0.000005,...,549.874733,0.0,0.0,False,True,True,0.0,0.0,0.0,0.0
4,pumpCmXqMfrsAkQ5r49WcJnRayYRqmXz6ae8H7H9Dfn,,1.752513e+09,2026-06-23 08:50:56+00:00,3ZnV54aAvQoHsjgWCHWKVk5ZaZxhuFe1UQ5AsaiDBHJnVP...,ARu4n5mFdZogZAravu7CcizaojWnS6oqka37gdLT5SZn,4vuV4WFwkMyuCfGbyDyZPTP5tFHYkxzwj2zFHtBfqR8N,0.000000,9.111748,0.000005,...,549.874733,0.0,0.0,False,True,True,0.0,0.0,0.0,0.0


In [ ]:
if not globals().get("QUALITY_GATE_PASSED", False):
    raise RuntimeError("MODEL FAILED QUALITY GATE — DO NOT EXPORT")

import onnxruntime as ort


def set_force_dropout(module: nn.Module, enabled: bool) -> None:
    for child in module.modules():
        if isinstance(child, MCDropout):
            child.force_dropout = enabled

id_to_deployer = {int(idx): deployer for deployer, idx in checkpoint["deployer_vocab"].items()}
with open(CONFIG["DEPLOYER_LOOKUP_PATH"], "w", encoding="utf-8") as handle:
    json.dump(id_to_deployer, handle, indent=2)

model.eval()
set_force_dropout(model, True)
example = next(iter(test_loader))
example_tabular = example["tabular"][:1].to(DEVICE)
example_sequence = example["sequence"][:1].to(DEVICE)
example_deployer = example["deployer_id"][:1].to(DEVICE)

try:
    torch.onnx.export(
        model,
        (example_tabular, example_sequence, example_deployer),
        CONFIG["ONNX_OUTPUT_PATH"],
        input_names=["tabular", "sequence", "deployer_id"],
        output_names=["rug_prob", "time_to_rug", "max_drawdown", "pump_2x"],
        dynamic_axes={
            "tabular": {0: "batch"},
            "sequence": {0: "batch"},
            "deployer_id": {0: "batch"},
            "rug_prob": {0: "batch"},
            "time_to_rug": {0: "batch"},
            "max_drawdown": {0: "batch"},
            "pump_2x": {0: "batch"},
        },
        opset_version=CONFIG["EXPORT_OPSET"],
        do_constant_folding=False,
    )
finally:
    set_force_dropout(model, False)

with open(CONFIG["MODEL_META_PATH"], "w", encoding="utf-8") as handle:
    json.dump(model_meta, handle, indent=2)

ort_session = ort.InferenceSession(CONFIG["ONNX_OUTPUT_PATH"], providers=["CPUExecutionProvider"])
ort_outputs = ort_session.run(
    None,
    {
        "tabular": example_tabular.detach().cpu().numpy().astype(np.float32),
        "sequence": example_sequence.detach().cpu().numpy().astype(np.float32),
        "deployer_id": example_deployer.detach().cpu().numpy().astype(np.int64),
    },
)
print({
    "onnx_path": CONFIG["ONNX_OUTPUT_PATH"],
    "deployer_lookup": CONFIG["DEPLOYER_LOOKUP_PATH"],
    "meta": CONFIG["MODEL_META_PATH"],
    "output_shapes": [np.asarray(output).shape for output in ort_outputs],
})
# Export XGBoost to ONNX
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_types = [('tabular_input', FloatTensorType([None, 14]))]
xgb_onnx = convert_sklearn(xgb_clf, initial_types=initial_types, target_opset=15)
with open(CONFIG['XGB_MODEL_OUT'], 'wb') as f:
    f.write(xgb_onnx.SerializeToString())
print(f"XGBoost ONNX exported to {CONFIG['XGB_MODEL_OUT']}")

# Validate XGB ONNX
xgb_sess = ort.InferenceSession(CONFIG['XGB_MODEL_OUT'], providers=['CPUExecutionProvider'])
test_tab = X_test_tabular[:1].astype(np.float32)
xgb_onnx_out = xgb_sess.run(None, {'tabular_input': test_tab})
xgb_onnx_prob = float(xgb_onnx_out[1][0][1])
assert 0 <= xgb_onnx_prob <= 1, f'XGB ONNX output out of range: {xgb_onnx_prob}'
print(f"XGB ONNX validation passed. Sample rug_prob: {xgb_onnx_prob:.4f}")

print({"xgb_onnx_path": CONFIG['XGB_MODEL_OUT'], "xgb_onnx_prob": xgb_onnx_prob})


## Deployment Checklist

[ ] Test AUC >= 0.65 (quality gate passed)
[ ] `rug_model.onnx` downloaded from Kaggle output
[ ] `deployer_lookup.json` downloaded and placed in `models/`
[ ] `models/rug_model_meta.json` updated with new `trained_at`, `train_period`, `val_period`, `test_period`, `test_auc`
[ ] `ALLOW_ONNX_RUG_MODEL=true` set in `.env` (was disabled pending retraining — this is now re-enabled)
[ ] `RUG_MODEL_PATH=models/rug_model.onnx` set in `.env`
[ ] Bot restarted and `rug_prob` scores logged for first 100 tokens to verify live inference is sane